In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / "projects" / "poreallas"
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import os
from datetime import UTC, datetime

import numpy as np
import xarray as xr
from dotenv import find_dotenv, load_dotenv

from utils import analysis_utils

importlib.reload(analysis_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'utils.analysis_utils' from '/home/emily_zuetell/projects/poreallas/utils/analysis_utils.py'>

In [ ]:
load_dotenv(
    find_dotenv("october_local.env", raise_error_if_not_found=True), override=True
)

### Set this to the current projection Month (e.g. 10: October forecast)
PROJ_MONTH = 10
MONTHS = [(PROJ_MONTH - 1 + i) % 12 + 1 for i in range(6)]
# Baseline period (Verify: 1996-2025)
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")

RATE = False

# Define Forecast
EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]  ## Use October Effects

# Open Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)

In [3]:
config = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",  # Set version to today
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=RATE,
    months=MONTHS,  # Forecast Months (6 months)
    hotonly="hotonly",  # We are using only hot-only for the tracker
    dims=["number", "sample"],
)

In [4]:
impact = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in config.dims}), config, ensemble=True
)
impact = impact.sel(month=config.months)

# Aggregate Subset of Impact Regions to group_level
impact = impact.sel(number=1, sample=1).sum("month")
impact, merge_key, base_cols = analysis_utils.aggregate_impact(impact, config, "ADM1")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped


In [10]:
expected = np.array(
    [-4.79437654e-04, 5.56102657e00, -1.66297806e-03, 1.85945622e01, 4.47127855e-01]
)
test = np.testing.assert_allclose(impact.head().values.ravel(), expected, rtol=1e-7)
print(f"{'pass' if test == None else 'fail'}")

pass
